In [ ]:
import os, json, random, shutil
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
WORK       = Path("/kaggle/working")

YOUR_COCO_JSON = "/kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json"   # Path("/kaggle/input/my-pcb/instances_default.json")
YOUR_IMG_ROOT  = "/kaggle/input/datasets/vidishagar/pcb-large-dataset/categories"   # Path("/kaggle/input/my-pcb/categories")
NEG_IMG_ROOT   = "/kaggle/input/datasets/vidishagar/non-def-1200/OK PHOTOS_MERGED"   # Path("/kaggle/input/my-negatives")

KEEP = ["Component Liftup","Component Missing","Component No Solder",
        "Component Solder Dry","Polarity Wrong","RYB Wrong Sequence",
        "Solder Ball","Solder Short"]

# RT-DETR model (Ultralytics). rtdetr-l = large; rtdetr-x = extra-large.
MODEL       = "rtdetr-l.pt"
IMG_SIZE    = 640
BATCH       = 8          # RT-DETR is heavier than YOLO; 8 is T4-safe. Lower to 4 if OOM.
EPOCHS      = 150
SEED        = 42
VAL_FRAC    = 0.20
N_NEG_TOTAL = 150

IMAGE_EXTS = [".jpg",".jpeg",".png",".bmp",".JPG",".JPEG",".PNG",".BMP"]
random.seed(SEED)

In [2]:
def coco_bbox_to_yolo(b,W,H):
    x,y,w,h=b
    return (x+w/2)/W,(y+h/2)/H,w/W,h/H

def write_yolo_split(coco, ids, split, out_root, src_lookup, names):
    (out_root/f"images/{split}").mkdir(parents=True,exist_ok=True)
    (out_root/f"labels/{split}").mkdir(parents=True,exist_ok=True)
    cats=sorted(coco["categories"],key=lambda c:c["id"])
    cid2yolo={c["id"]:i for i,c in enumerate(cats)}
    byid={im["id"]:im for im in coco["images"]}
    anns=defaultdict(list)
    for a in coco["annotations"]: anns[a["image_id"]].append(a)
    ni=nb=0
    for i in ids:
        im=byid[i]; src=src_lookup(im["file_name"])
        if src is None: continue
        uniq=im["file_name"].replace("/","__").replace("\\","__")
        shutil.copy(src, out_root/f"images/{split}/{uniq}")
        W,H=im["width"],im["height"]; lines=[]
        for a in anns.get(i,[]):
            cx,cy,w,h=coco_bbox_to_yolo(a["bbox"],W,H)
            lines.append(f"{cid2yolo[a['category_id']]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            nb+=1
        (out_root/f"labels/{split}/{Path(uniq).stem}.txt").write_text("\n".join(lines))
        ni+=1
    return ni,nb

def add_negatives(neg_paths, out_root, n_total, val_frac, seed=SEED):
    if not neg_paths: print("no negatives"); return
    negs=neg_paths[:]; random.Random(seed).shuffle(negs); negs=negs[:n_total]
    nval=round(n_total*val_frac)
    for split,chunk in (("val",negs[:nval]),("train",negs[nval:])):
        (out_root/f"images/{split}").mkdir(parents=True,exist_ok=True)
        (out_root/f"labels/{split}").mkdir(parents=True,exist_ok=True)
        for p in chunk:
            shutil.copy(p, out_root/f"images/{split}/{p.name}")
            (out_root/f"labels/{split}/{p.stem}.txt").write_text("")
    print(f"negatives: {len(negs)-nval} train, {nval} val")

def filter_coco_to_classes(coco, keep_names):
    keep_set=set(keep_names)
    id2name={c["id"]:c["name"] for c in coco["categories"]}
    new_cats=[{"id":i+1,"name":n} for i,n in enumerate(keep_names)]
    name2new={c["name"]:c["id"] for c in new_cats}
    new_anns=[]
    for a in coco["annotations"]:
        nm=id2name.get(a["category_id"])
        if nm in keep_set:
            b=dict(a); b["category_id"]=name2new[nm]; new_anns.append(b)
    imgs_with_ann={a["image_id"] for a in new_anns}
    new_imgs=[im for im in coco["images"] if im["id"] in imgs_with_ann]
    return {"images":new_imgs,"annotations":new_anns,"categories":new_cats}

In [3]:
if YOUR_COCO_JSON is None:
    for c in INPUT_ROOT.rglob("*.json"):
        try:
            d=json.loads(Path(c).read_text())
            if "annotations" in d and "categories" in d:
                YOUR_COCO_JSON=c; break
        except Exception: pass
print("YOUR_COCO_JSON =", YOUR_COCO_JSON)
assert YOUR_COCO_JSON, "Set YOUR_COCO_JSON in CELL 0."

your_coco=json.loads(Path(YOUR_COCO_JSON).read_text())
YOUR_NAMES=[c["name"] for c in sorted(your_coco["categories"],key=lambda c:c["id"])]
print(f"{len(YOUR_NAMES)} classes in file:",YOUR_NAMES)

if YOUR_IMG_ROOT is None: YOUR_IMG_ROOT=Path(YOUR_COCO_JSON).parent
IMG_BASE=Path(YOUR_IMG_ROOT).parent
your_src={}
for p in Path(YOUR_IMG_ROOT).rglob("*"):
    if p.suffix.lower() in (".jpg",".jpeg",".png",".bmp"):
        your_src[str(p.relative_to(IMG_BASE))]=p
print("your images found:",len(your_src))

_bidx=defaultdict(list)
for k,v in your_src.items(): _bidx[Path(k).name].append(v)
def resolve_your_image(fn):
    if fn in your_src: return your_src[fn]
    h=_bidx.get(Path(fn).name,[]); return h[0] if len(h)==1 else None

found=sum(1 for im in your_coco["images"] if resolve_your_image(im["file_name"]))
print(f"COCO resolvable: {found}/{len(your_coco['images'])}")
assert found>=len(your_coco["images"])*0.98

if NEG_IMG_ROOT is None:
    for r in (INPUT_ROOT.iterdir() if INPUT_ROOT.exists() else []):
        if any(k in r.name.lower() for k in ("neg","good","clean","ok","defect-free")):
            NEG_IMG_ROOT=r; break
neg_files=[p for p in Path(NEG_IMG_ROOT).rglob("*")
           if p.suffix.lower() in (".jpg",".jpeg",".png",".bmp")] if NEG_IMG_ROOT else []
print("negatives:",len(neg_files),"from",NEG_IMG_ROOT)

YOUR_COCO_JSON = /kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json
12 classes in file: ['Component Crack', 'Component Damage', 'Component Liftup', 'Component Missing', 'Component No Solder', 'Component Solder Dry', 'LED Damage', 'Polarity Wrong', 'RYB Wrong Sequence', 'Solder Ball', 'Solder Short', 'Tombstone']
your images found: 551
COCO resolvable: 548/551
negatives: 1195 from /kaggle/input/datasets/vidishagar/non-def-1200/OK PHOTOS_MERGED


In [4]:
missing=[k for k in KEEP if k not in YOUR_NAMES]
assert not missing, f"KEEP names not in YOUR_NAMES: {missing}"
coco8 = filter_coco_to_classes(your_coco, KEEP)
NAMES8 = [c["name"] for c in sorted(coco8["categories"],key=lambda c:c["id"])]
id2n = {c["id"]:c["name"] for c in coco8["categories"]}
print(f"8-class: {len(coco8['images'])} images, {len(coco8['annotations'])} anns")
cc=Counter(a["category_id"] for a in coco8["annotations"])
for cid in sorted(cc): print(f"  {id2n[cid]:<22} {cc[cid]}")

8-class: 508 images, 565 anns
  Component Liftup       22
  Component Missing      44
  Component No Solder    114
  Component Solder Dry   138
  Polarity Wrong         36
  RYB Wrong Sequence     24
  Solder Ball            43
  Solder Short           144


In [5]:
cls_freq=Counter(a["category_id"] for a in coco8["annotations"])
img_cls=defaultdict(set)
for a in coco8["annotations"]: img_cls[a["image_id"]].add(a["category_id"])
strat=defaultdict(list)
for im in coco8["images"]:
    cls=img_cls.get(im["id"],set())
    key=min(cls,key=lambda c:cls_freq[c]) if cls else "__none__"
    strat[key].append(im["id"])
rng=random.Random(SEED); y_train=set(); y_val=set()
for k,ids in strat.items():
    ids=ids[:]; rng.shuffle(ids)
    nv=max(1,round(len(ids)*VAL_FRAC)) if len(ids)>1 else 0
    y_val|=set(ids[:nv]); y_train|=set(ids[nv:])
assert not (y_train & y_val)
print(f"train {len(y_train)} | val {len(y_val)}")
for c in sorted(cls_freq):
    ntr=sum(1 for a in coco8["annotations"] if a["category_id"]==c and a["image_id"] in y_train)
    nva=sum(1 for a in coco8["annotations"] if a["category_id"]==c and a["image_id"] in y_val)
    print(f"  {id2n[c]:<22} train {ntr:>3} val {nva:>3}")

train 407 | val 101
  Component Liftup       train  18 val   4
  Component Missing      train  35 val   9
  Component No Solder    train  97 val  17
  Component Solder Dry   train 111 val  27
  Polarity Wrong         train  29 val   7
  RYB Wrong Sequence     train  19 val   5
  Solder Ball            train  35 val   8
  Solder Short           train 114 val  30


In [6]:
import yaml
DATA=WORK/"yolo8"; shutil.rmtree(DATA, ignore_errors=True)
ysl=lambda n: resolve_your_image(n)
print("pos train:",write_yolo_split(coco8,y_train,"train",DATA,ysl,NAMES8))
print("pos val  :",write_yolo_split(coco8,y_val,  "val",  DATA,ysl,NAMES8))
add_negatives(neg_files,DATA,N_NEG_TOTAL,VAL_FRAC)
(DATA/"data.yaml").write_text(yaml.safe_dump(
    {"path":str(DATA),"train":"images/train","val":"images/val",
     "names":{i:n for i,n in enumerate(NAMES8)}},sort_keys=False))
print("data.yaml written; classes:",NAMES8)

pos train: (404, 453)
pos val  : (101, 107)
negatives: 120 train, 30 val
data.yaml written; classes: ['Component Liftup', 'Component Missing', 'Component No Solder', 'Component Solder Dry', 'Polarity Wrong', 'RYB Wrong Sequence', 'Solder Ball', 'Solder Short']


In [7]:
for split in ["train","val"]:
    imgs=[p for p in (DATA/f"images/{split}").rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png",".bmp")]
    labs=list((DATA/f"labels/{split}").rglob("*.txt"))
    nonempty=[p for p in labs if p.read_text().strip()]
    print(f"[{split}] images={len(imgs)} labels={len(labs)} non-empty={len(nonempty)}")

[train] images=524 labels=524 non-empty=404
[val] images=131 labels=131 non-empty=101


In [8]:
!pip install ultralytics

In [9]:
from ultralytics import RTDETR
model = RTDETR(MODEL)
model.train(
    data=str(DATA/"data.yaml"),
    epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, patience=30,
    project=str(WORK/"runs"), name="rtdetr8_640",
    device=[0,1],
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15, shear=10, scale=0.5,
    fliplr=0.5, flipud=0.5, mosaic=1.0,
    lr0=0.0001,               # RT-DETR uses a lower LR than YOLO (AdamW-style)
    seed=SEED,
)
BEST = str(WORK/"runs/rtdetr8_640/weights/best.pt")
print("best ->", BEST)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo8/data.yaml, degrees=15, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr8_640

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      1/150      3.78G      1.184      30.39      1.091          1        640: 100% ━━━━━━━━━━━━ 66/66 1.7it/s 37.9s0.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.6it/s 3.5s0.2ss
                   all        131        107   7.79e-05     0.0462   1.42e-05   5.78e-06

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      2/150      3.97G     0.7762      1.919     0.6081          4        640: 100% ━━━━━━━━━━━━ 66/66 2.4it/s 27.7s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 6.0it/s 1.5s0.2s
                   all        131        107    0.00132      0.212     0.0042    0.00258

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

    126/150      3.97G     0.3522     0.4673     0.2144          4        640: 100% ━━━━━━━━━━━━ 66/66 2.4it/s 27.2s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 5.8it/s 1.6s0.2s
                   all        131        107      0.784      0.726      0.765       0.52

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

    127/150      3.97G     0.3012      0.446     0.2118          1        640: 100% ━━━━━━━━━━━━ 66/66 2.4it/s 27.1s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 5.7it/s 1.6s0.2s
                   all        131        107      0.761      0.719      0.763      0.501

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

    128/150      3.97G     0.3133     0.4486      0.179          1        640: 100% ━━━━━━━━━━━━ 66/66 2.4it/s 27.3s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 5.8it/s 1.6s0.2s
                   all        131        107      0.778      0.749      0.766       0.51
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 98, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

128 epochs completed in 1.068 hours.
Optimizer stripped from /kaggle/working/runs/rtdetr8_640-2/weights/last.pt, 66.3MB
Optimizer stripped from /kaggle/working/runs/rtdetr8_640-2/weights/best.pt, 66.3MB

Validating /kaggle/working/runs/rtdetr8_640-2/weights/best.pt...
rt-detr-l summary: 310 layers, 32,000,180 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Image

In [10]:
res = model.val(data=str(DATA/"data.yaml"), split="val")
print(f"\n=== RT-DETR 8-CLASS (640px, with negatives) ===")
print(f"mAP@0.5      {res.box.map50:.4f}")
print(f"mAP@0.5:0.95 {res.box.map:.4f}")
print(f"precision    {res.box.mp:.4f}")
print(f"recall       {res.box.mr:.4f}")
print("\nper-class mAP@0.5:")
for i,n in res.names.items():
    m=res.box.maps[i] if i<len(res.box.maps) else 0
    print(f"  {n:<22} {m:.3f}")
print("\nCompare against your YOLO baseline: 0.757 (single-run) / 0.76 ± 0.02 (CV)")

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 310 layers, 32,000,180 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2614.0±283.6 MB/s, size: 236.7 KB)
val: Scanning /kaggle/working/yolo8/labels/val.cache... 131 images, 30 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 131/131 26.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.7it/s 5.3s0.6ss
                   all        131        107      0.807      0.749       0.77      0.556
      Component Liftup          4          4      0.762          1      0.796      0.625
     Component Missing          9          9      0.971      0.889      0.984      0.753
   Component No Solder         16         17      0.852       0.34      0.391      0.315
  Component Solder Dry         24         27      0.667      0.815      0.776      0.563
        Polarity Wrong          7        